# EuroCoin Training Notebook

Minimal OOP training scaffold:
- Stage 1: YOLO detector
- Stage 2: ResNet classifier on cropped detections
- Stage 3: baseline classifier on cropped detections

It is written to work both on a local PC and in Google Drive / Colab, as long as the `eurocoin-vision` project folder exists or `EUROCOIN_VISION_ROOT` is set.

In [ ]:
# Uncomment in a fresh environment if needed:
# %pip install -q ultralytics torch torchvision pyyaml pillow

import copy
import os
import random
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import ResNet18_Weights, resnet18
from ultralytics import YOLO
import yaml


@dataclass(frozen=True)
class ProjectPaths:
    project_root: Path

    @property
    def ml_pipeline_dir(self) -> Path:
        return self.project_root / "ml_pipeline"

    @property
    def datasets_dir(self) -> Path:
        return self.ml_pipeline_dir / "datasets"

    @property
    def runs_dir(self) -> Path:
        return self.ml_pipeline_dir / "runs"

    @property
    def model_weights_dir(self) -> Path:
        return self.project_root / "model_weights"

    @property
    def classification_dir(self) -> Path:
        return self.ml_pipeline_dir / "classification_datasets"

    def stage_dataset_dir(self, stage_name: str) -> Path:
        return self.datasets_dir / stage_name

    def stage_weight_dir(self, stage_name: str) -> Path:
        return self.model_weights_dir / stage_name


class RuntimeEnvironment:
    @staticmethod
    def is_colab() -> bool:
        return "google.colab" in sys.modules


class GoogleDriveMountService:
    def __init__(self, mount_point: Path = Path("/content/drive")) -> None:
        self._mount_point = mount_point

    @property
    def my_drive_dir(self) -> Path:
        return self._mount_point / "MyDrive"

    def ensure_mounted(self) -> None:
        if not RuntimeEnvironment.is_colab():
            return

        if self.my_drive_dir.exists():
            return

        from google.colab import drive

        print("Mounting Google Drive...")
        drive.mount(str(self._mount_point), force_remount=False)
        if not self.my_drive_dir.exists():
            raise FileNotFoundError(
                f"Google Drive mount finished, but {self.my_drive_dir} was not found."
            )


class ProjectLocator:
    def __init__(self, project_name: str = "eurocoin-vision") -> None:
        self._project_name = project_name
        self._drive_mount_service = GoogleDriveMountService()

    def find(self) -> ProjectPaths:
        self._drive_mount_service.ensure_mounted()

        checked_candidates: list[Path] = []
        for candidate in self._candidate_roots():
            checked_candidates.append(candidate)
            if candidate.exists() and self._looks_like_project_root(candidate):
                return ProjectPaths(candidate)

        searched_roots: list[Path] = []
        for search_root in self._recursive_search_roots():
            if not search_root.exists():
                continue

            searched_roots.append(search_root)
            for candidate in self._discover_project_roots(search_root):
                if self._looks_like_project_root(candidate):
                    return ProjectPaths(candidate)

        raise FileNotFoundError(
            "Could not find the eurocoin-vision project root. "
            f"Checked direct candidates: {[str(path) for path in checked_candidates]} | "
            f"Recursive search roots: {[str(path) for path in searched_roots]}"
        )

    def _candidate_roots(self) -> list[Path]:
        candidates: list[Path] = []
        env_root = os.getenv("EUROCOIN_VISION_ROOT")
        if env_root:
            candidates.append(Path(env_root).expanduser())

        cwd = Path.cwd().resolve()
        candidates.extend([cwd, *cwd.parents])
        candidates.extend([
            cwd / self._project_name,
            Path.home() / self._project_name,
            Path.home() / "Desktop" / "Git" / "Projects" / self._project_name,
            Path.home() / "Documents" / self._project_name,
            Path("/content") / self._project_name,
            self._drive_mount_service.my_drive_dir / self._project_name,
            Path("/content") / "drive" / "MyDrive" / self._project_name,
            Path("/content") / "drive" / "MyDrive" / "Colab Notebooks" / self._project_name,
            Path("/content") / "drive" / "MyDrive" / "Projects" / self._project_name,
        ])

        unique_candidates: list[Path] = []
        seen: set[Path] = set()
        for candidate in candidates:
            resolved = candidate.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            unique_candidates.append(resolved)

        return unique_candidates

    def _recursive_search_roots(self) -> list[Path]:
        roots = [
            self._drive_mount_service.my_drive_dir,
            self._drive_mount_service.my_drive_dir / "Colab Notebooks",
            self._drive_mount_service.my_drive_dir / "Projects",
            Path("/content"),
        ]

        unique_roots: list[Path] = []
        seen: set[Path] = set()
        for root in roots:
            resolved = root.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            unique_roots.append(resolved)
        return unique_roots

    def _discover_project_roots(self, search_root: Path, max_depth: int = 6) -> list[Path]:
        discovered: list[Path] = []

        for current_root, dir_names, _ in os.walk(search_root):
            current_path = Path(current_root)
            try:
                depth = len(current_path.relative_to(search_root).parts)
            except ValueError:
                depth = 0

            dir_names[:] = [
                directory_name
                for directory_name in dir_names
                if not directory_name.startswith(".")
            ]

            if current_path.name == self._project_name:
                discovered.append(current_path)
                dir_names[:] = []
                continue

            if self._project_name in dir_names:
                discovered.append(current_path / self._project_name)

            if depth >= max_depth:
                dir_names[:] = []

        return discovered

    def _looks_like_project_root(self, path: Path) -> bool:
        return (path / "ml_pipeline").exists()


def seed_everything(seed: int = 52) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


paths = ProjectLocator().find()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(52)

print(f"Project root: {paths.project_root}")
print(f"Device: {device}")


In [ ]:
@dataclass(frozen=True)
class YoloTrainerConfig:
    model_name: str = "yolov8n.pt"
    epochs: int = 50
    imgsz: int = 640
    batch: int = 16
    workers: int = 2
    device: str | int = "cuda" if torch.cuda.is_available() else "cpu"
    run_name: str = "stage1_yolo"


class Stage1YoloTrainer:
    def __init__(self, paths: ProjectPaths, config: YoloTrainerConfig | None = None) -> None:
        self._paths = paths
        self._config = config or YoloTrainerConfig()

    def train(self) -> Path:
        data_yaml = self._paths.stage_dataset_dir("stage1") / "data.yaml"
        if not data_yaml.exists():
            raise FileNotFoundError(f"Stage 1 dataset not found: {data_yaml}")

        run_project = self._paths.runs_dir / "stage1"
        run_project.mkdir(parents=True, exist_ok=True)

        model = YOLO(self._config.model_name)
        model.train(
            data=str(data_yaml),
            project=str(run_project),
            name=self._config.run_name,
            epochs=self._config.epochs,
            imgsz=self._config.imgsz,
            batch=self._config.batch,
            workers=self._config.workers,
            device=self._config.device,
        )

        best_weights = run_project / self._config.run_name / "weights" / "best.pt"
        if not best_weights.exists():
            raise FileNotFoundError(f"YOLO best.pt not found: {best_weights}")

        publish_dir = self._paths.stage_weight_dir("stage1")
        publish_dir.mkdir(parents=True, exist_ok=True)
        publish_path = publish_dir / f"{self._config.run_name}_best.pt"
        shutil.copy2(best_weights, publish_path)

        best_model = YOLO(str(best_weights))
        test_metrics = best_model.val(
            data=str(data_yaml),
            split="test",
            imgsz=self._config.imgsz,
            batch=self._config.batch,
            device=self._config.device,
        )
        print("Stage 1 test metrics:")
        for metric_name, metric_value in test_metrics.results_dict.items():
            print(f"- {metric_name}: {metric_value}")

        print(f"Stage 1 published to: {publish_path}")
        return publish_path


In [ ]:
class DetectionCropDatasetBuilder:
    def __init__(self, detection_stage_dir: Path, output_dir: Path) -> None:
        self._detection_stage_dir = detection_stage_dir
        self._output_dir = output_dir

    @property
    def output_dir(self) -> Path:
        return self._output_dir

    def build(self) -> Path:
        class_names = self._load_class_names()
        if self._output_dir.exists():
            shutil.rmtree(self._output_dir)

        self._prepare_class_directories(class_names)

        for split_name in ("train", "val", "test"):
            images_dir = self._detection_stage_dir / "images" / split_name
            labels_dir = self._detection_stage_dir / "labels" / split_name
            if not images_dir.exists() or not labels_dir.exists():
                raise FileNotFoundError(f"Missing split directory in {self._detection_stage_dir}: {split_name}")

            for image_path in sorted(path for path in images_dir.iterdir() if path.is_file()):
                label_path = labels_dir / f"{image_path.stem}.txt"
                if not label_path.exists():
                    continue
                self._export_crops(
                    image_path=image_path,
                    label_path=label_path,
                    split_name=split_name,
                    class_names=class_names,
                )

        return self._output_dir

    def _load_class_names(self) -> list[str]:
        data_yaml_path = self._detection_stage_dir / "data.yaml"
        if not data_yaml_path.exists():
            raise FileNotFoundError(f"Missing data.yaml: {data_yaml_path}")

        data = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
        names = data["names"]
        if isinstance(names, dict):
            return [names[index] for index in sorted(names)]
        return list(names)

    def _prepare_class_directories(self, class_names: list[str]) -> None:
        for split_name in ("train", "val", "test"):
            for class_name in class_names:
                (self._output_dir / split_name / class_name).mkdir(parents=True, exist_ok=True)

    def _export_crops(
        self,
        image_path: Path,
        label_path: Path,
        split_name: str,
        class_names: list[str],
    ) -> None:
        image = Image.open(image_path).convert("RGB")
        image_width, image_height = image.size

        for object_index, raw_line in enumerate(label_path.read_text(encoding="utf-8").splitlines()):
            line = raw_line.strip()
            if not line:
                continue

            class_id_text, x_center_text, y_center_text, width_text, height_text = line.split()
            class_id = int(class_id_text)
            class_name = class_names[class_id]

            left, top, right, bottom = self._to_pixel_box(
                x_center=float(x_center_text),
                y_center=float(y_center_text),
                box_width=float(width_text),
                box_height=float(height_text),
                image_width=image_width,
                image_height=image_height,
            )

            if right <= left or bottom <= top:
                continue

            class_dir = self._output_dir / split_name / class_name
            class_dir.mkdir(parents=True, exist_ok=True)
            crop = image.crop((left, top, right, bottom))
            crop.save(class_dir / f"{image_path.stem}_{object_index}.png")

    def _to_pixel_box(
        self,
        x_center: float,
        y_center: float,
        box_width: float,
        box_height: float,
        image_width: int,
        image_height: int,
    ) -> tuple[int, int, int, int]:
        box_width_pixels = box_width * image_width
        box_height_pixels = box_height * image_height
        center_x_pixels = x_center * image_width
        center_y_pixels = y_center * image_height

        left = max(0, int(center_x_pixels - box_width_pixels / 2))
        top = max(0, int(center_y_pixels - box_height_pixels / 2))
        right = min(image_width, int(center_x_pixels + box_width_pixels / 2))
        bottom = min(image_height, int(center_y_pixels + box_height_pixels / 2))
        return left, top, right, bottom


In [ ]:
@dataclass(frozen=True)
class ClassifierTrainerConfig:
    run_name: str
    epochs: int = 5
    batch_size: int = 32
    image_size: int = 224
    learning_rate: float = 1e-3
    num_workers: int = 2
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class ImageFolderDataModule:
    def __init__(self, dataset_root: Path, config: ClassifierTrainerConfig) -> None:
        self._dataset_root = dataset_root
        self._config = config
        self.class_names: list[str] = []

    def build_loaders(self) -> tuple[DataLoader, DataLoader, DataLoader, list[str]]:
        train_transform = transforms.Compose([
            transforms.Resize((self._config.image_size, self._config.image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
        ])
        val_transform = transforms.Compose([
            transforms.Resize((self._config.image_size, self._config.image_size)),
            transforms.ToTensor(),
        ])

        train_dataset = ImageFolder(self._dataset_root / "train", transform=train_transform)
        val_dataset = ImageFolder(self._dataset_root / "val", transform=val_transform)
        test_dataset = ImageFolder(self._dataset_root / "test", transform=val_transform)
        self.class_names = train_dataset.classes

        train_loader = DataLoader(
            train_dataset,
            batch_size=self._config.batch_size,
            shuffle=True,
            num_workers=self._config.num_workers,
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=self._config.batch_size,
            shuffle=False,
            num_workers=self._config.num_workers,
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=self._config.batch_size,
            shuffle=False,
            num_workers=self._config.num_workers,
        )
        return train_loader, val_loader, test_loader, self.class_names


class TorchvisionClassifierTrainer:
    def __init__(self, paths: ProjectPaths, stage_name: str, config: ClassifierTrainerConfig) -> None:
        self._paths = paths
        self._stage_name = stage_name
        self._config = config

    def fit(self, dataset_root: Path) -> Path:
        data_module = ImageFolderDataModule(dataset_root=dataset_root, config=self._config)
        train_loader, val_loader, test_loader, class_names = data_module.build_loaders()

        model = self._build_model(num_classes=len(class_names)).to(self._config.device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=self._config.learning_rate)

        best_accuracy = 0.0
        best_state = copy.deepcopy(model.state_dict())

        for epoch in range(self._config.epochs):
            train_loss = self._train_one_epoch(model, train_loader, criterion, optimizer)
            val_loss, val_accuracy = self._evaluate(model, val_loader, criterion)
            print(
                f"{self._stage_name} | epoch {epoch + 1}/{self._config.epochs} | "
                f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_accuracy:.4f}"
            )

            if val_accuracy >= best_accuracy:
                best_accuracy = val_accuracy
                best_state = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_state)
        test_loss, test_accuracy = self._evaluate(model, test_loader, criterion)
        print(
            f"{self._stage_name} | test_loss={test_loss:.4f} | test_acc={test_accuracy:.4f}"
        )

        publish_dir = self._paths.stage_weight_dir(self._stage_name)
        publish_dir.mkdir(parents=True, exist_ok=True)
        publish_path = publish_dir / f"{self._config.run_name}_best.pt"
        torch.save(
            {
                "model_state_dict": best_state,
                "class_names": class_names,
                "image_size": self._config.image_size,
                "best_val_accuracy": best_accuracy,
                "test_accuracy": test_accuracy,
                "test_loss": test_loss,
            },
            publish_path,
        )
        print(f"{self._stage_name} published to: {publish_path}")
        return publish_path

    def _build_model(self, num_classes: int) -> nn.Module:
        model = resnet18(weights=ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    def _train_one_epoch(
        self,
        model: nn.Module,
        loader: DataLoader,
        criterion: nn.Module,
        optimizer: torch.optim.Optimizer,
    ) -> float:
        model.train()
        total_loss = 0.0

        for images, labels in loader:
            images = images.to(self._config.device)
            labels = labels.to(self._config.device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)

        return total_loss / max(1, len(loader.dataset))

    def _evaluate(
        self,
        model: nn.Module,
        loader: DataLoader,
        criterion: nn.Module,
    ) -> tuple[float, float]:
        model.eval()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        with torch.no_grad():
            for images, labels in loader:
                images = images.to(self._config.device)
                labels = labels.to(self._config.device)

                logits = model(images)
                loss = criterion(logits, labels)
                predictions = logits.argmax(dim=1)

                total_loss += loss.item() * images.size(0)
                total_correct += (predictions == labels).sum().item()
                total_samples += images.size(0)

        mean_loss = total_loss / max(1, total_samples)
        accuracy = total_correct / max(1, total_samples)
        return mean_loss, accuracy


class Stage2ResNetTrainer(TorchvisionClassifierTrainer):
    def __init__(self, paths: ProjectPaths, config: ClassifierTrainerConfig | None = None) -> None:
        super().__init__(
            paths=paths,
            stage_name="stage2",
            config=config or ClassifierTrainerConfig(run_name="stage2_resnet18"),
        )


class Stage3BaselineTrainer(TorchvisionClassifierTrainer):
    def __init__(self, paths: ProjectPaths, config: ClassifierTrainerConfig | None = None) -> None:
        super().__init__(
            paths=paths,
            stage_name="stage3",
            config=config or ClassifierTrainerConfig(run_name="stage3_resnet18"),
        )


In [ ]:
# Run prepare_datasets.py first so that ml_pipeline/datasets/stage1|stage2|stage3 exist.

In [ ]:
class WorkingDirectory:
    def __init__(self, target_dir: Path) -> None:
        self._target_dir = target_dir
        self._previous_dir: Path | None = None

    def __enter__(self) -> Path:
        self._previous_dir = Path.cwd()
        os.chdir(self._target_dir)
        print(f"Working directory: {self._target_dir}")
        return self._target_dir

    def __exit__(self, exc_type, exc_value, traceback) -> None:
        if self._previous_dir is not None:
            os.chdir(self._previous_dir)


class FullTrainingPipeline:
    def __init__(self, paths: ProjectPaths) -> None:
        self._paths = paths

    def run(
        self,
        prepare_datasets_first: bool = True,
        run_stage1: bool = True,
        run_stage2: bool = True,
        run_stage3: bool = True,
    ) -> dict[str, Path]:
        outputs: dict[str, Path] = {}

        if prepare_datasets_first:
            self._prepare_datasets()

        if run_stage1:
            outputs["stage1"] = self._run_stage1()

        if run_stage2:
            outputs["stage2"] = self._run_stage2()

        if run_stage3:
            outputs["stage3"] = self._run_stage3()

        return outputs

    def _prepare_datasets(self) -> None:
        print("[1/4] Preparing datasets...")
        import importlib.util

        module_path = self._paths.ml_pipeline_dir / "prepare_datasets.py"
        spec = importlib.util.spec_from_file_location("prepare_datasets_module", module_path)
        if spec is None or spec.loader is None:
            raise ImportError(f"Could not load dataset module from {module_path}")

        module = importlib.util.module_from_spec(spec)
        with WorkingDirectory(self._paths.ml_pipeline_dir):
            spec.loader.exec_module(module)
            module.main()

    def _run_stage1(self) -> Path:
        print("[2/4] Training Stage 1 YOLO...")
        with WorkingDirectory(self._paths.stage_dataset_dir("stage1")):
            trainer = Stage1YoloTrainer(
                self._paths,
                YoloTrainerConfig(
                    epochs=18,
                    run_name="stage1_yolov8n",
                ),
            )
            return trainer.train()

    def _run_stage2(self) -> Path:
        print("[3/4] Building Stage 2 crops and training ResNet...")
        with WorkingDirectory(self._paths.stage_dataset_dir("stage2")):
            stage2_crop_root = DetectionCropDatasetBuilder(
                detection_stage_dir=self._paths.stage_dataset_dir("stage2"),
                output_dir=self._paths.classification_dir / "stage2_material",
            ).build()
            trainer = Stage2ResNetTrainer(
                self._paths,
                ClassifierTrainerConfig(
                    run_name="stage2_resnet18",
                    epochs=5,
                    batch_size=32,
                ),
            )
            return trainer.fit(stage2_crop_root)

    def _run_stage3(self) -> Path:
        print("[4/4] Building Stage 3 crops and training classifier...")
        with WorkingDirectory(self._paths.stage_dataset_dir("stage3")):
            stage3_crop_root = DetectionCropDatasetBuilder(
                detection_stage_dir=self._paths.stage_dataset_dir("stage3"),
                output_dir=self._paths.classification_dir / "stage3_denomination",
            ).build()
            trainer = Stage3BaselineTrainer(
                self._paths,
                ClassifierTrainerConfig(
                    run_name="stage3_resnet18",
                    epochs=5,
                    batch_size=32,
                ),
            )
            return trainer.fit(stage3_crop_root)


pipeline = FullTrainingPipeline(paths)

# Main one-cell run.
# Disable stages by switching True -> False if needed.
artifacts = pipeline.run(
    prepare_datasets_first=True,
    run_stage1=True,
    run_stage2=True,
    run_stage3=True,
)

print("\nTraining artifacts:")
for stage_name, artifact_path in artifacts.items():
    print(f"- {stage_name}: {artifact_path}")
